In [7]:
import os

import pandas as pd
import truststore

truststore.inject_into_ssl()

"AWS_ACCESS_KEY_ID" in os.environ, "AWS_SECRET_ACCESS_KEY" in os.environ

(True, True)

In [8]:
import s3fs

fs = s3fs.S3FileSystem(
    endpoint_url="https://s3.ru-7.storage.selcloud.ru",
    client_kwargs={"region_name": "ru-7"},
)
fs.ls("hse-recsys-project")

['hse-recsys-project/data']

In [9]:
fashion = pd.read_csv("../reports/eda_amazon_fashion.csv")
grocery = pd.read_csv("../reports/eda_grocery_and_gourmet_food.csv")

cmp = fashion.merge(
    grocery,
    on="проверка",
    how="right",
    suffixes=("_fashion", "_grocery"),
    sort=False,
)
cmp[["проверка", "факт_fashion", "ok_fashion", "факт_grocery", "ok_grocery"]]

,проверка,факт_fashion,ok_fashion,факт_grocery,ok_grocery
0,колонки и типы как у Fashion,NaN,NaN,True,True
1,пропуски в ключевых колонках,0,True,0,True
2,rating только 1-5,0,True,1,False
3,timestamp совпадает с year,0,True,0,True
4,helpful_vote >= 0,0,True,1,False
5,полные дубли строк,25244,False,130928,False
6,повторные отзывы юзера на товар,1.06%,False,1.66%,False
7,повторы после удаления дублей,0.05%,True,0.75%,True
8,parent_asin в meta уникален,0,True,0,True
9,товар отзыва есть в meta,100.00%,True,100.00%,True


по качеству данных категории похожи: обе чистые, ботов нет, пропусков в ключевых колонках нет, у товаров почти всегда есть название и картинки. отличаются они по насыщенности: у Grocery юзеров с одним товаром 66.5% против 86.2% у Fashion, теплых товаров 42.9% против 11.3%, description у 62.7% товаров против 7.2%. у Fashion провал весной 2020 (-38.6%), у Grocery падения не больше -19.3%. единственный пункт, где Fashion лучше: разница контента у холодных и теплых товаров 8.8 п.п. против 12.3 п.п.

In [10]:
import pyarrow.dataset as ds

BASE = "hse-recsys-project/data/processed"
CATEGORIES = ["amazon_fashion", "grocery_and_gourmet_food"]


def load_pairs(category):
    path = f"{BASE}/reviews/{category}"
    d = ds.dataset(path, filesystem=fs, partitioning="hive")
    t = d.to_table(columns=["user_id", "parent_asin"]).to_pandas()
    return t.drop_duplicates()


pairs = {c: load_pairs(c) for c in CATEGORIES}
{c: len(p) for c, p in pairs.items()}

{'amazon_fashion': 2474375, 'grocery_and_gourmet_food': 14081170}

In [11]:
def k_core(p, k=5):
    while True:
        u = p.groupby("user_id")["parent_asin"].transform("size")
        i = p.groupby("parent_asin")["user_id"].transform("size")
        keep = (u >= k) & (i >= k)
        if keep.all():
            return p
        p = p[keep]


rows = []
for c, p in pairs.items():
    core = k_core(p)
    rows.append(
        {
            "категория": c,
            "юзеров": core["user_id"].nunique(),
            "товаров": core["parent_asin"].nunique(),
            "пар": len(core),
            "доля пар": f"{len(core) / len(p):.1%}",
        }
    )
pd.DataFrame(rows)

,категория,юзеров,товаров,пар,доля пар
0,amazon_fashion,0,0,0,0.0%
1,grocery_and_gourmet_food,404760,132865,3948741,28.0%


In [12]:
rows = []
for k in [2, 3, 4, 5]:
    row = {"k": k}
    for c, p in pairs.items():
        row[c] = len(k_core(p, k))
    rows.append(row)
pd.DataFrame(rows)

,k,amazon_fashion,grocery_and_gourmet_food
0,2,289897,9182081
1,3,12490,6620453
2,4,3805,5038940
3,5,0,3948741


## k-core:

k-core оставляет только юзеров, у которых хотя бы k товаров, и товары, у которых хотя бы k юзеров. при k = 5 у Fashion не осталось ни одного взаимодействия, а у Grocery осталось 3 948 741 (28% от всех пар): 404 760 юзеров и 132 865 товаров. у Fashion уже при k = 3 осталось всего 0.5% пар, у Grocery 47%

## Итог сравнения:

обе категории чистые: пропусков в ключевых колонках нет, ботов нет, у товаров почти всегда есть название и картинки. отличаются они по тому, сколько у юзеров взаимодействий. у Fashion 86.2% юзеров купили один товар, теплых товаров 11.3%, после фильтра 5-core не остается ни одного взаимодействия. у Grocery один товар у 66.5% юзеров, теплых товаров 42.9%, после 5-core остается 3 948 741 взаимодействие. еще у Grocery описание есть у 62.7% товаров, у Fashion у 7.2%.